# 第3回　確率論の基礎(2)：ベイズの定理の深掘り
## ―― 「データで信念を更新する」。直感を裏切る基準率の誤謬

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

前回（第2回）の条件付き確率を **逆向き** に使うのが、今日のベイズの定理だ。▶ を上から押して、直感が見事に外れる様子を確認しよう。

今日のキーワード：**ベイズの定理／基準率／陽性的中率／逐次更新**。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK。次のセルへ。")

---
## 1. 直感クイズ ―― 検査が陽性、あなたは病気か？

ある病気の検査がある。性能はとても良い：

- 病気の人を正しく陽性と判定する確率（**感度**）：99%
- 健康な人を正しく陰性と判定する確率（**特異度**）：99%

この病気は、100人に **1人**（有病率 1%）がかかっている稀な病気だ。

あなたは検査で **陽性** になった。

**問い：あなたが本当に病気である確率は？**

99%の精度だから……99%くらい？　予想を決めてから ▶。

In [ ]:
# 自然頻度で考える：100万人を検査したら何人がどうなる？
人口 = 1_000_000
有病率 = 0.01           # 100人に1人
感度 = 0.99            # P(陽性 | 病気)
特異度 = 0.99          # P(陰性 | 健康)

病気の人 = 人口 * 有病率
健康な人 = 人口 * (1 - 有病率)

真陽性 = 病気の人 * 感度              # 病気で陽性
偽陽性 = 健康な人 * (1 - 特異度)       # 健康なのに陽性

陽性的中率 = 真陽性 / (真陽性 + 偽陽性)   # P(病気 | 陽性)

print(f"100万人のうち病気の人： {病気の人:>10,.0f} 人")
print(f"  → うち陽性（真陽性）： {真陽性:>10,.0f} 人")
print(f"健康な人　　　　　　 ： {健康な人:>10,.0f} 人")
print(f"  → うち陽性（偽陽性）： {偽陽性:>10,.0f} 人")
print()
print(f"陽性になった人 {真陽性+偽陽性:,.0f} 人のうち、本当に病気は {真陽性:,.0f} 人")
print(f"→ P(病気 | 陽性) ＝ {陽性的中率:.1%}")

**答えは99%ではなく、約 50%。** 検査が陽性でも、病気である確率はコイン投げと変わらない。

理由：病気の人はそもそも極端に少ない（1000人に1人）。だから「健康なのに誤って陽性（偽陽性）」になる人が、「本当に病気で陽性」の人と同じくらい出てしまう。**検査の精度（99%）と、陽性的中率（50%）はまったく別の数字**だ。

> 直感は「P(陽性\|病気)=99%」を「P(病気\|陽性)=99%」と取り違える。これが第2回でやった **条件付き確率の向きの混同**であり、今日の最重要ポイント。

---
## 2. ベイズの定理

いま計算したことを式にすると、**ベイズの定理** になる。

$$P(\text{病気}\mid\text{陽性}) = \frac{P(\text{陽性}\mid\text{病気})\,P(\text{病気})}{P(\text{陽性})}$$

一般形ではこう書く。

$$\underbrace{P(H\mid D)}_{\text{事後確率}} = \frac{\overbrace{P(D\mid H)}^{\text{尤度}}\;\overbrace{P(H)}^{\text{事前確率}}}{P(D)}$$

- **事前確率** $P(H)$：データを見る前の信念（ここでは有病率 0.1%）
- **尤度** $P(D\mid H)$：その仮説のもとでデータが出る確率（感度 99%）
- **事後確率** $P(H\mid D)$：データを見た後に更新された信念（陽性的中率 50%）

**ベイズの定理＝データで信念を更新する計算**。事前 0.1% が、陽性というデータで 50% に更新された。

---
## 3. 基準率（有病率）を動かすと、答えは激変する

同じ性能（感度99%・特異度99%）の検査でも、**病気の珍しさ（基準率）** が変われば陽性的中率はまるで違う。

予想：有病率が高い病気（例：10%）なら、陽性的中率はどうなる？

In [ ]:
def 陽性的中率を計算(有病率, 感度=0.99, 特異度=0.99):
    真陽性 = 有病率 * 感度
    偽陽性 = (1 - 有病率) * (1 - 特異度)
    return 真陽性 / (真陽性 + 偽陽性)

有病率リスト = np.array([0.0001, 0.001, 0.01, 0.05, 0.1, 0.3, 0.5])
for p in 有病率リスト:
    print(f"有病率 {p:>6.2%} → 陽性的中率 P(病気|陽性) = {陽性的中率を計算(p):>6.1%}")

xs = np.linspace(0.0001, 0.5, 400)
plt.figure(figsize=(7, 4))
plt.plot(xs, [陽性的中率を計算(p) for p in xs], color="#3949ab")
plt.scatter([0.01], [陽性的中率を計算(0.01)], color="#e8503a", zorder=5,
            label="有病率1% → 約50%")
plt.xlabel("有病率（基準率）")
plt.ylabel("陽性的中率 P(病気 | 陽性)")
plt.title("同じ検査でも、基準率で陽性的中率は激変する")
plt.legend()
plt.show()

珍しい病気ほど、陽性的中率は低い。**検査の性能は何も変わっていないのに、だ。**

直感がこれを外すのは、**基準率（その事象がもともとどれくらい起こりやすいか）を無視する**から。これを **基準率の誤謬（base rate fallacy）** と呼ぶ。

---
## 4. 逐次更新 ―― 今日の事後は、明日の事前

ベイズの真価は、**データが来るたびに信念を更新できる**ことだ。「今日の事後確率」を「明日の事前確率」として使い、新しいデータでまた更新する。

例：あるコインの「表が出る確率 $p$」を当てたい。最初は何も分からない（$p$ はどの値も等しくありそう＝一様な事前）。投げるたびに信念がどう締まっていくかを見よう。

（数学的には、事前をベータ分布 $\mathrm{Beta}(a,b)$ にすると、表 $k$ 回・裏 $n-k$ 回の観測で事後は $\mathrm{Beta}(a+k,\;b+n-k)$ になる。）

In [ ]:
# 本当は表が出る確率 0.7 のコインを、少しずつ投げて信念を更新する
rng = np.random.default_rng(3)
真のp = 0.7
投げる回数リスト = [0, 5, 20, 100]
全コイン = rng.random(100) < 真のp     # True=表 を100回ぶん用意

xs = np.linspace(0, 1, 500)
plt.figure(figsize=(8, 5))
for n in 投げる回数リスト:
    k = 全コイン[:n].sum()                 # n回中の表の数
    事後 = stats.beta(1 + k, 1 + (n - k))  # 一様事前 Beta(1,1) からの更新
    plt.plot(xs, 事後.pdf(xs), label=f"{n}回投げた後（表{int(k)}回）")
plt.axvline(真のp, ls="--", color="gray", label="本当のp=0.7")
plt.xlabel("表が出る確率 p についての信念")
plt.ylabel("事後分布の密度")
plt.title("データが増えるほど、信念は真の値の周りに締まる（ベイズ更新）")
plt.legend()
plt.show()

最初（0回）は真っ平ら＝「$p$ は何も分からない」。投げる回数が増えるほど、事後分布は **本当の値 0.7 の周りに鋭く締まっていく**。

これが **学習** の数理的な姿だ。事前の思い込みがあっても、データを積めば信念は正しい方へ更新されていく。

> 第12〜13回では逆に、**人々が互いを真似て同じデータばかり見ると、集団の信念が間違った方向に固まってしまう**（情報カスケード）。ベイズ更新が健全に働くにも、やはり独立で多様な情報が要る。

---
## 今日のまとめ

| 用語 | 意味 |
|---|---|
| ベイズの定理 | $P(H\mid D)=\dfrac{P(D\mid H)\,P(H)}{P(D)}$。データで信念を更新する |
| 事前 / 尤度 / 事後 | 見る前の信念 / データの出やすさ / 更新後の信念 |
| 基準率の誤謬 | 事象のもともとの珍しさ（基準率）を無視する誤り |
| 陽性的中率 | P(病気\|陽性)。検査の精度とは別物 |

- 99%精度の検査でも、稀な病気では陽性的中率は約50%。**精度と的中率は別**。
- 原因は **基準率の無視** と、**条件付き確率の向きの混同**（P(陽性\|病気) ≠ P(病気\|陽性)）。
- ベイズは「今日の事後＝明日の事前」でデータを積むほど信念が締まる＝学習の数理。

> **課題（Moodle）**：陽性的中率の計算（自動採点）＋「なぜ直感は基準率を無視するのか／ベイズ更新を自分の言葉で」の記述。詳しくはMoodleの第3回課題を見ること。

> **次回予告**：第4回「離散確率分布：二項分布・ポアソン分布」。「10回投げて7回表。イカサマ？」 ―― 独立な試行をn回くりかえすと、結果はどんな分布になるか。